# Using the ZMART-controller

The controller offers one short, fixed list of commands, and every microscope answers them in the same way. A workflow written against these commands runs on any microscope that has a ZMART driver. This notebook goes through every command once, on the mock driver: a simulated microscope that comes with the controller, so nothing here needs hardware. Run the cells from top to bottom.

## Connect

`get_instruments` lists the drivers installed on this computer, with how each one reaches its microscope. The mock is always there. `connect` opens the connection; from then on, every command is `mic.<command>`. The second argument is optional and goes to the driver unchanged; here it tells the mock where to save its images.

In [ ]:
from zmart_controller import mic

mic.get_instruments()

In [ ]:
mic.connect("mock", {"output_root": "images"})

## Meet the microscope

Every command answers with two things: `success`, whether the driver did what you asked, and `content`, what it has to say. The content of `get_info` is a description of the microscope in plain words: what each setting means, its unit and bounds, which objectives are fitted. Read it once when you meet a new microscope; it is what the other commands cannot tell you.

In [ ]:
print(mic.get_info()["content"]["description"])

## Where is the stage

Positions are in micrometres from the origin, a point that was chosen and saved once for this microscope, so a position means the same place on the sample every time you connect. Each axis has one or more motors; the mock's z has a coarse one and a piezo for fine steps. The `canvas` of an axis is everywhere a picture can show: the travel of the stage plus half a field of view. Plan your positions inside it.

In [ ]:
mic.get_actuators()["content"]

In [ ]:
mic.get_xyz()["content"]

## Move

`set_xyz` takes all three axes, so a script always says exactly where it wants to be. When the answer comes back, the stage has arrived. A move outside the travel is not done: the answer is `success: False`, and `content` says why. A workflow should look at `success` before it carries on.

In [ ]:
mic.set_xyz(100, 50, 0)["content"]

In [ ]:
mic.set_xyz(99999, 0, 0)

## Settings

The state is a snapshot of the settings. Its `changeable` part is what `set_state` can apply: here laser power, gain, exposure and the objective. Send back some or all of them; the rest stay as they are. The driver reads each setting back to make sure it took, and the answer names what was applied. A state is a plain dictionary, so you can save it with `json` and apply it again another day.

In [ ]:
mic.get_state()["content"]["changeable"]

In [ ]:
mic.set_state({"changeable": {"exposure_ms": 20.0}})["content"]

## Take a picture

Acquisition settings are the choices about how to capture and save, as opposed to the microscope's own settings: the file format, the planes of a z-stack. For each one, `options` says what it may be and `active` what is used when you say nothing.

`acquire` captures at the current position with the current settings and saves, in one step. The label names this position in the saved files. The answer lists every file saved under `files`, and under `planes` says for every image plane which file, channel, depth and moment it is and where on the sample it was taken. Use those paths rather than building your own: where the files go is up to the driver.

In [ ]:
mic.get_acquisition_settings()["content"]

In [ ]:
answer = mic.acquire(position_label="A1")
answer["content"]["files"]

A z-stack is one acquisition with several planes. The mock starts at the current height and goes up; every plane records the height it was taken at.

In [ ]:
answer = mic.acquire(position_label="A2", acquisition_settings={"z_planes": 3, "z_step_um": 2.0})
[(plane["z"], plane["z_um"]) for plane in answer["content"]["planes"]]

## Run a routine

Procedures are routines the microscope offers, such as autofocus. Each driver offers its own; the description says what a routine does and which options it takes. Run one by name, with its options beside the name.

In [ ]:
mic.get_procedures()["content"]

In [ ]:
mic.run_procedure({"name": "autofocus", "range_um": 10, "step_um": 1})["content"]

## Disconnect

Close the connection when you are done. To drive the microscope again, call `connect` again.

In [ ]:
mic.disconnect()

---

MIT license. Thom de Hoog, Center for Microscopy and Image Analysis (ZMB), University of Zurich. thom.dehoog@zmb.uzh.ch, thomdehoog@gmail.com.